# Treinando modelos com o MLFlow no Projeto

Neste projeto de encerramento do módulo, consolidaremos o uso do **MLflow** e do **Optuna** para treinar e comparar múltiplos modelos de classificação de comportamento de compra.

### Definição da Regra de Negócio e Variável Alvo (Target)
Para alimentar os algoritmos de classificação, precisamos transformar os logs brutos de eventos em um problema supervisionado. Criaremos a coluna **`comprou_eletronico`** com base nas seguintes condições da nossa tabela Gold:
* **Target = 1:** Se o usuário realizou uma conversão (`tipo_evento == 'finalizar_compra'`) dentro da seção de tecnologia (`categoria == 'Eletrônicos'`).
* **Target = 0:** Caso contrário (visualizações, cliques, ou compras de outras categorias).


In [0]:
%%capture
%pip install databricks
%pip install xgboost lightgbm optuna -q
%pip install shap -q
dbutils.library.restartPython()

In [0]:
from pyspark.sql.functions import col, avg, sum, first, lower, trim, to_date
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

# Configuração de Escopos e Nomes das Tabelas
catalogo = "workspace"
esquema = "gold"
tabela_origem = f"{catalogo}.{esquema}.base_eventos_produto_gold_particionada"
nome_feature_table = f"{catalogo}.{esquema}.base_features_modelo"
tabela_observacao_nome = f"{catalogo}.{esquema}.base_propensao_compra_observacao"

df_gold = spark.table(tabela_origem)

# para salvar no Feature Store do databricks a coluna precisa ser do tipo DATE, timestamp.
df_gold_com_data = df_gold.withColumn("dia_prtc", to_date(col("dia_prtc").cast("string"), "yyyyMMdd"))

df_user_features_temporal = df_gold_com_data.groupBy("id_usuario", "dia_prtc").agg(
    first("idade").alias("idade_cliente"),
    lower(trim(first("genero"))).alias("genero_cliente"),
    lower(trim(first("regiao"))).alias("regiao_cliente"),
    first("renda_mensal_estimada_k").alias("renda_mensal_k"),
    avg("tempo_clique_segundos").alias("tempo_medio_clique_segundos"),
    avg("interacoes_chat_suporte").alias("media_interacoes_suporte"),
    avg("cupons_ativos_conta").alias("media_cupons_ativos"),
    avg("score_satisfacao_nps").alias("media_score_nps_cliente"),
    avg("dias_desde_ultima_visita").alias("media_dias_inatividade"),
    sum(col("preco_reais")).alias("total_gasto_acumulado_reais")
)



In [0]:
df_user_features_temporal.display()

Databricks Feature Store: fe.create_table

O método `fe.create_table` do `FeatureEngineeringClient` é o comando utilizado para registrar e estruturar uma nova **Feature Table** gerenciada dentro do Unity Catalog.

* **`name`**: O caminho completo de três níveis no Unity Catalog (`catalogo.esquema.nome_da_tabela`) onde a Feature Table Delta será fisicamente criada e catalogada.
* **`primary_keys`**: Lista com as colunas que identificam de forma única cada registro. No exemplo, a chave é composta por `id_usuario` e `dia_prtc`.
* **`timeseries_columns`**: Identifica explicitamente qual coluna representa o carimbo de data. É o parâmetro mais importante para habilitar a **consistência histórica (Point-in-time correctness)**, garantindo que o modelo nunca consulte dados do futuro durante o treinamento.
* **`df`**: O DataFrame do Spark (ou PySpark) contendo os dados e os recursos que foram calculados nas etapas anteriores de engenharia de dados.
* **`description`**: Texto livre descritivo. Esse metadado fica visível na interface do Unity Catalog para que outros cientistas de dados entendam o propósito e a origem dessas variáveis.


In [0]:
fe = FeatureEngineeringClient()

fe.create_table(
    name=nome_feature_table,
    primary_keys=["id_usuario", "dia_prtc"],
    timeseries_columns=["dia_prtc"],
    df=df_user_features_temporal,
    description="Recursos comportamentais agregados por usuário."
)


## Tabela de Observação e Criação de Target

Este bloco de código cria a **Tabela de Observação**, o ponto de partida obrigatório para o treinamento utilizando o `FeatureEngineeringClient`. Esta tabela funciona como o "esqueleto" do dataset final: ela armazena apenas as chaves de amarração, o marcador de tempo e a variável resposta (**Target**).

1. **`select("id_usuario", ...)`**: Filtra as colunas da camada Gold necessárias para mapear o comportamento.
2. **`withColumn("comprou_eletronico", when(...))`**: Aplica a regra de negócio para construir o target de classificação binária. O Spark avalia se, naquele exato registro, o usuário estava na categoria correta **E** executou a ação de conversão. Se verdadeiro, atribui `1`, caso contrário, `0`.
3. **`select("id_usuario", "timestamp_registro", "comprou_eletronico")`**: Limpa o DataFrame, mantendo **apenas** o identificador do cliente, o momento do evento e o resultado do target. Isolar essas colunas aqui impede o vazamento de dados (*data leakage*) no modelo.
4. **`saveAsTable(tabela_observacao_nome)`**: Persiste a estrutura resultante como uma tabela Delta no Unity Catalog.

### Por que a Feature Table não tem a Target?
* **Feature Table:** Guarda apenas o histórico comportamental do usuário (médias, somatórios, características). Ela é imutável e reutilizável por múltiplos modelos.
* **Tabela de Observação:** Define o escopo específico do seu problema de negócio atual (quem comprou o quê e quando). O `FeatureLookup` usará o `id_usuario` e o `timestamp_registro` desta tabela para buscar o estado do cliente no passado exato daquele evento.

In [0]:
from pyspark.sql.functions import col, when, to_date

df_observacao = df_gold.withColumn("dia_prtc", to_date(col("dia_prtc").cast("string"), "yyyyMMdd")) \
    .withColumn("comprou_eletronico", when(
        (col("categoria") == "Eletrônicos") & (col("tipo_evento") == "finalizar_compra"), 1
                                              ).otherwise(0)
    )\
                                                  .select("id_usuario","dia_prtc","comprou_eletronico")

In [0]:
df_observacao.write.mode("overwrite").saveAsTable(tabela_observacao_nome)

## FeatureLookup e create_training_set

Este bloco realiza a **amarração e o cruzamento inteligente** dos dados. Ele utiliza as definições de busca para construir a matriz de treino final, garantindo o alinhamento temporal exato de cada variável antes de converter o resultado para Pandas.

#### `FeatureLookup`

* **`feature_names`**: Lista com os nomes exatos das colunas calculadas que você quer extrair da Feature Table.
* **`lookup_key`**: A chave primária de amarração (`id_usuario`). O Spark usará esse ID para saber de qual cliente ele deve buscar as métricas.
* **`timestamp_lookup_key`**: A coluna de tempo da tabela de observação. Ela força o motor da Feature Store a buscar apenas o estado das variáveis que o usuário tinha **no momento ou antes** daquele carimbo de data/hora, mitigando o vazamento de dados do futuro.

#### `fe.create_training_set`

* **`df`**: O DataFrame contendo a sua tabela de observação (Chaves + Target).
* **`feature_lookups`**: A lista de objetos `FeatureLookup` configurados acima.
* **`label`**: Indica qual coluna é a variável alvo (`comprou_eletronico`). O Databricks armazena essa informação nos metadados para saber o que o modelo tentará prever.
* **`exclude_columns`**: Colunas que devem ser descartadas da matriz final de entrada do modelo. IDs e timestamps são removidos aqui para que o algoritmo dependa exclusivamente das features comportamentais para tomar a decisão.

#### `.load_df().toPandas()`

* O método `.load_df()` resolve o plano de execução e expõe a matriz montada como um DataFrame Spark distribuído. O `.toPandas()` deixa dataset no formato nativo exigido pelo Scikit-Learn e Optuna.


In [0]:
recursos_desejados = [
    FeatureLookup(
        table_name=nome_feature_table,
        feature_names=['idade_cliente',
 'genero_cliente',
 'regiao_cliente',
 'renda_mensal_k',
 'tempo_medio_clique_segundos',
 'media_interacoes_suporte',
 'media_cupons_ativos',
 'media_score_nps_cliente',
 'media_dias_inatividade',
 'total_gasto_acumulado_reais']
            ,
        lookup_key="id_usuario",
        timestamp_lookup_key="dia_prtc"
    )
]

dataset_treino = fe.create_training_set(
    df=df_observacao,
    feature_lookups=recursos_desejados,
    label="comprou_eletronico",
    exclude_columns=["id_usuario","dia_prtc"]
)

# Carrega como DataFrame local do Pandas
df_final_pandas = dataset_treino.load_df().toPandas()
df_final_pandas.display()

## Treinamento de Modelos Utilizando MLFlow

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split

# garantindo que as colunas venham como numericas
df_final_pandas["renda_mensal_k"] = pd.to_numeric(df_final_pandas["renda_mensal_k"], errors='coerce').fillna(0).astype(float)
df_final_pandas["total_gasto_acumulado_reais"] = pd.to_numeric(df_final_pandas["total_gasto_acumulado_reais"], errors='coerce').fillna(0).astype(float)

# Tratamento de variáveis categóricas
df_projeto_encoded = pd.get_dummies(
    df_final_pandas, 
    columns=["genero_cliente", "regiao_cliente"], 
    drop_first=True
)
df_projeto_encoded.display()

# Divisão de Features (X) e Target (y)
X = df_projeto_encoded.drop(columns=["comprou_eletronico"])
y = df_projeto_encoded["comprou_eletronico"]

# Divisão de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
import mlflow
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

# Dicionário de modelos de baseline
modelos_baseline = {
    "Logistic_Regression_Base": LogisticRegression(max_iter=1000, random_state=42),
    "Random_Forest_Base": RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42),
    "Gradient_Boosting_Base": GradientBoostingClassifier(n_estimators=50, random_state=42),
    "XGBoost_Base": XGBClassifier(n_estimators=50, max_depth=5, random_state=42, eval_metric='logloss'),
    "LightGBM_Base": LGBMClassifier(n_estimators=50, max_depth=5, random_state=42, verbosity=-1)
}

print("Iniciando treinamento dos modelos")

for nome_modelo, classificador in modelos_baseline.items():
    with mlflow.start_run(run_name=nome_modelo):
        
        # Treino do modelo
        classificador.fit(X_train, y_train)
        
        # Previsão
        y_pred = classificador.predict_proba(X_test)[:,1]
        
        # Cálculo das métricas
        auc_score = roc_auc_score(y_test, y_pred)
        acc_score = accuracy_score(y_test, y_pred.round())
        
        
        # Registro no MLflow
        mlflow.log_param("algoritmo", classificador.__class__.__name__)
        mlflow.log_metric("auc_roc", auc_score)
        mlflow.log_metric("accuracy", acc_score)
        
        print(f"{nome_modelo} -> AUC-ROC: {auc_score:.4f} | Acurácia: {acc_score:.4f}")

print("\nTodos os baselines salvos no MLflow!")

In [0]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
import mlflow

def objetivo_optuna_lgbm(trial):
    # Espaço de busca específico para a arquitetura do LightGBM
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 12)
    num_leaves = trial.suggest_int('num_leaves', 10, 100)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    min_child_samples = trial.suggest_int('min_child_samples', 5, 50)
    
    with mlflow.start_run(run_name=f"Optuna_LGBM_Trial_{trial.number}", nested=True):
        
        # Log manual dos parâmetros sugeridos neste trial
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("num_leaves", num_leaves)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("min_child_samples", min_child_samples)
        
        # Instanciação do classificador LightGBM
        # verbosity=-1 retira os avisos de info internos do LightGBM para não poluir o notebook
        clf = LGBMClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            num_leaves=num_leaves,
            learning_rate=learning_rate,
            min_child_samples=min_child_samples,
            random_state=42,
            verbosity=-1
        )
        
        # Treinamento local
        clf.fit(X_train, y_train)
        
        # Previsão das probabilidades para cálculo da AUC-ROC
        y_pred = clf.predict_proba(X_test)[:,1]
        
        # Cálculo da AUC-ROC
        auc = roc_auc_score(y_test, y_pred)
        
        # Registro da métrica no MLflow
        mlflow.log_metric("AUC-ROC", auc)

        return auc
        

print("Iniciando busca refinada de hiperparâmetros para o LightGBM com Optuna...")
estudo_lgbm = optuna.create_study(direction="maximize")

# Agrupando todas as tentativas sob a run do LightGBM no MLflow
with mlflow.start_run(run_name="Optuna_LGBM", nested=True):
    estudo_lgbm.optimize(objetivo_optuna_lgbm, n_trials=100)

print("\nTuning do LightGBM concluído!")
print(f"Melhor AUC-ROC obtida: {estudo_lgbm.best_value:.4f}")
print(f"Melhores hiperparâmetros encontrados: {estudo_lgbm.best_params}")

# Entendendo o `log_model` e a `Signature` no MLflow

## O que é a `Signature` (Assinatura)?

A **Signature** é o contrato de dados oficial do seu modelo. Ela define e congela o formato estrito que o modelo espera receber na entrada (**Inputs**) e o que ele vai devolver na saída (**Outputs**).

Sem uma assinatura cadastrada, o modelo fica vulnerável a falhas silenciosas de produção (ex: um aplicativo web enviar um dado de texto para uma coluna que deveria ser estritamente float).

### Como ela atua em tempo real?

Quando o Databricks Model Serving ou o pipeline de lote lê um modelo que possui uma assinatura, ele aplica o **Schema Enforcement**:

1. **Validação de Nomes:** Garante que todas as colunas exigidas estejam presentes na requisição JSON ou na tabela Delta.
2. **Validação de Tipagem:** Bloqueia a execução imediatamente se houver divergência estrita de tipos (ex: se o modelo espera `integer` nas variáveis binárias e o Spark tentar enviar `boolean`, o MLflow interrompe o processo blindando o LightGBM de quebras internas).
3. **Validação de Ordem:** Para engines de boosting distribuídas (como o LightGBM/XGBoost que utilizam matrizes matemáticas nativas em C++), a assinatura garante que as colunas sejam injetadas exatamente na ordem correta do treinamento.


In [0]:
import mlflow
from mlflow.models import infer_signature
from lightgbm import LGBMClassifier

catalogo_prod = "workspace"
esquema_prod = "gold"
model_name = f"{catalogo_prod}.{esquema_prod}.propensao_compra_model"

print("Recuperando os melhores parâmetros do Optuna que acabou de rodar...")
melhores_parametros = estudo_lgbm.best_params
print(f"Melhores parâmetros: {melhores_parametros}")

# Instancia e treina o modelo final com a base completa de treino
melhor_lgbm = LGBMClassifier(**melhores_parametros, random_state=42, verbosity=-1)
melhor_lgbm.fit(X_train, y_train)


signature = infer_signature(X_train, melhor_lgbm.predict(X_test))
# Criamos uma run dedicada para registrar o modelo oficial de produção
with mlflow.start_run(run_name="Modelo_Final_LGBM_Production"):
    
    # Logamos as métricas e parâmetros finais
    for param_name, param_val in melhores_parametros.items():
        mlflow.log_param(param_name, param_val)
        
    print(f"Registrando o modelo no Unity Catalog como: {model_name}...")
    fe.log_model(
        model=melhor_lgbm,
        artifact_path="model",
        flavor=mlflow.lightgbm,         # Aqui indicamos que flavor é o LightGBM
        registered_model_name=model_name,
        training_set=dataset_treino,     
        signature=signature            
    )
